# Session 6 (Day 3 — Afternoon)
## Group Simulation Challenge

GIS Executive Seminar — Enhancing Data Analysis Skills for Informed Investment Insights

---

## How This Session Works

| Phase | Duration | Activity |
|-------|----------|----------|
| Briefing | 30 min | Facilitator explains challenges; groups assigned; roles allocated |
| Build | 90 min | Groups work on their challenge using Claude Code |
| Break | 15 min | — |
| Presentations | 30 min | Each group presents (5 min each + Q&A) |
| Debrief | 15 min | Full group discussion |

## Group Roles
- **Investment Lead** — defines what the tool should do and what investment insight it should surface. Owns the investment thesis.
- **Claude Code Director** — writes the brief, iterates the instruction, reads the output. Does not need to understand the code — only the outcome.
- **QA / Interpreter** — checks the output for errors and interprets the numbers in investment terms. Catches anything that looks wrong.
- **Presenter** — structures the 5-minute presentation for the group.

## Three Challenges

| Challenge | Based On | Focus |
|-----------|----------|-------|
| **A** | `portfolio_app.py` | Extend the Portfolio Viewer Dash app |
| **B** | `momentum_investing.ipynb` | Interrogate the Momentum Strategy backtest |
| **C** | Starter code in this notebook | Build a Macro Regime Dashboard |

**Navigate to your group's section below.**

## Judging Criteria

Each group is scored 1–5 on five criteria:

| Criterion | What We Are Looking For |
|-----------|------------------------|
| **Functionality** | Does the app/notebook run without errors? Does the data pull correctly? |
| **Analytical Depth** | Are the metrics correctly computed and interpreted? |
| **Investment Insight** | Does the tool help a portfolio manager make a better decision? Is the investment thesis clear? |
| **Presentation** | Clear, confident, structured — appropriate for an investment committee |
| **Clarity of Direction** | How precisely did the group brief Claude Code? (assessed by facilitator during build phase) |

Maximum score: 25 points.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# COMMON SETUP — ALL GROUPS RUN THIS CELL FIRST
# This downloads all the data needed for all three challenges.
# It takes about 30 seconds.
# ══════════════════════════════════════════════════════════════════════════════

# Install all required libraries quietly
!pip install yfinance fredapi plotly scipy pandas numpy --quiet

import pandas as pd                          # data manipulation
import numpy as np                           # numerical computing
import plotly.graph_objects as go           # interactive charts
from plotly.subplots import make_subplots   # multi-panel dashboards
from fredapi import Fred                     # FRED macroeconomic data client
import yfinance as yf                        # Yahoo Finance price data
from datetime import date                    # date arithmetic
from scipy import stats                      # statistical functions
from IPython.display import display          # display DataFrames nicely in Colab

# ── GIS PORTFOLIO DEFINITION ──────────────────────────────────────────────────
PORTFOLIO_TICKERS = ['AAPL', 'MSFT', 'JPM', 'XOM', 'JNJ', 'GLD', 'IEF']   # the 7 GIS holdings
BENCHMARK_TICKER  = '^GSPC'                                                   # S&P 500 benchmark
ALL_TICKERS       = PORTFOLIO_TICKERS + [BENCHMARK_TICKER]                   # combined list for download
n_assets          = len(PORTFOLIO_TICKERS)                                   # 7 holdings
weights           = np.repeat(1 / n_assets, n_assets)                       # equal weight: each = 1/7

# ← CHANGE ME: paste your FRED API key here
FRED_API_KEY = 'YOUR_FRED_API_KEY'   # ← CHANGE ME
fred = Fred(api_key=FRED_API_KEY)    # create the FRED client

# ── DATE RANGE ────────────────────────────────────────────────────────────────
end_date    = date.today()                                                     # today
start_date  = date(end_date.year - 3, end_date.month, end_date.day)          # 3 years of equity data
macro_start = date(end_date.year - 5, end_date.month, end_date.day)          # 5 years for macro context

# ── PRICE DATA FROM YFINANCE ──────────────────────────────────────────────────
print('Downloading equity prices...')
raw_data   = yf.download(                   # download price data
    tickers     = ALL_TICKERS,
    start       = start_date,
    end         = end_date,
    auto_adjust = True,                      # adjust for splits and dividends
    progress    = True,
)
all_prices = raw_data['Close'].ffill()       # extract closes; forward-fill gaps
prices     = all_prices[PORTFOLIO_TICKERS]   # portfolio holdings only
benchmark  = all_prices[[BENCHMARK_TICKER]]  # S&P 500 only

# ── RETURNS ───────────────────────────────────────────────────────────────────
daily_returns          = prices.pct_change().dropna()                        # daily returns for each holding
benchmark_daily        = benchmark.pct_change().dropna().squeeze()           # S&P 500 daily returns
portfolio_daily_return = daily_returns.dot(weights)                          # equal-weighted portfolio return

# ── CUMULATIVE RETURNS ────────────────────────────────────────────────────────
cumulative_portfolio  = (1 + portfolio_daily_return).cumprod().mul(100)     # portfolio grows from 100
cumulative_individual = (1 + daily_returns).cumprod().mul(100)              # each holding grows from 100
cumulative_benchmark  = (1 + benchmark_daily).cumprod().mul(100)            # benchmark grows from 100

# ── RISK METRICS ──────────────────────────────────────────────────────────────
ann_return  = portfolio_daily_return.mean() * 252                           # annualised return
ann_vol     = portfolio_daily_return.std() * np.sqrt(252)                   # annualised volatility
running_max = cumulative_portfolio.cummax()                                  # rolling peak value
drawdown    = cumulative_portfolio.div(running_max).sub(1)                   # % below prior peak
max_dd      = drawdown.min()                                                  # worst drawdown

var_95      = np.percentile(portfolio_daily_return, 5)                      # 95% VaR (1-day)
tail_losses = portfolio_daily_return[portfolio_daily_return <= var_95]      # returns below VaR threshold
cvar_95     = tail_losses.mean()                                             # average of tail losses = CVaR

monthly_returns = (                                                           # monthly returns for correlation
    prices
    .resample('ME')
    .last()
    .pct_change()
    .dropna()
)
corr_matrix = monthly_returns.corr().round(2)                               # correlation matrix

# ── MACRO DATA FROM FRED ──────────────────────────────────────────────────────
print('Downloading macro data from FRED...')
try:
    spread_series    = fred.get_series('T10Y2Y',   observation_start=macro_start, observation_end=end_date).dropna()
    cpi_headline     = fred.get_series('CPIAUCSL', observation_start=macro_start, observation_end=end_date)
    cpi_core         = fred.get_series('CPILFESL', observation_start=macro_start, observation_end=end_date)
    fed_funds        = fred.get_series('DFF',      observation_start=macro_start, observation_end=end_date).dropna()
    unemployment     = fred.get_series('UNRATE',   observation_start=macro_start, observation_end=end_date).dropna()
    usrec            = fred.get_series('USREC',    observation_start=macro_start, observation_end=end_date)

    cpi_headline_yoy = cpi_headline.pct_change(12).dropna().mul(100)        # headline CPI year-on-year %
    cpi_core_yoy     = cpi_core.pct_change(12).dropna().mul(100)            # core CPI year-on-year %

    rec_starts       = usrec[(usrec == 1) & (usrec.shift(1) == 0)].index   # recession start dates
    rec_ends         = usrec[(usrec == 0) & (usrec.shift(1) == 1)].index   # recession end dates

    macro_ok = True
    print('FRED data downloaded successfully.')
except Exception as e:
    macro_ok = False
    print(f'FRED data unavailable: {e}')
    print('Challenge C requires FRED data. Challenges A and B will still work.')

# ── SUMMARY ───────────────────────────────────────────────────────────────────
print()
print('=== SETUP COMPLETE ===')
print(f'  Portfolio:      {PORTFOLIO_TICKERS}')
print(f'  Date range:     {start_date}  →  {end_date}')
print(f'  Trading days:   {len(portfolio_daily_return)}')
print(f'  Ann. Return:    {ann_return:.2%}')
print(f'  Ann. Vol:       {ann_vol:.2%}')
print(f'  Max Drawdown:   {max_dd:.2%}')
print(f'  VaR (95%, 1d):  {var_95:.2%}')
print(f'  CVaR (95%, 1d): {cvar_95:.2%}')
print()
print('Navigate to your group\'s challenge section below.')

---
---

# ══════════════════════════════════════
# CHALLENGE A — Extend the Portfolio Viewer
# ══════════════════════════════════════

## What `portfolio_app.py` Already Does

`portfolio_app.py` is a fully working two-tab Dash application:

**Tab 1 — Performance & Risk:**
- Cumulative return chart (portfolio vs S&P 500, base 100, in USD)
- CAPM regression table: alpha, beta, R², p-value, number of monthly observations
- Rolling 6-month annualised volatility chart

**Tab 2 — FX & Regional Analysis:**
- Weighted cumulative returns by currency bucket in local currency terms
- USD return decomposition: local return + FX impact = total USD return
- Regional summary table (return, volatility, weight by region)
- Per-holding detail table (ticker, name, weight, local return, FX return, USD return)

**How to run it locally:**
```
python portfolio_app.py
# Then open: http://127.0.0.1:8050
```

## Your Challenge — Pick ONE Extension

Direct Claude Code to add one of these to the app:

**Option 1: Add VaR/CVaR/Drawdown to Tab 1**
Below the CAPM table, add a Dash DataTable with three rows: `95% Historical VaR (1-day)`, `95% CVaR (1-day)`, `Maximum Drawdown`. Format as percentages with 2 decimal places. Colour negative values red.

**Option 2: Add a third tab — Macro Context**
Pull 3 FRED series (2Y/10Y spread, CPI YoY, Fed Funds Rate) and display them as styled metric cards alongside the portfolio cumulative return chart.

**Option 3: Add a drawdown chart to Tab 1**
Add the underwater equity curve (% decline from prior peak) for the portfolio and the S&P 500. Show the date of the maximum drawdown as an annotation.

## Suggested Claude Code Brief (Option 1)

Copy this brief and give it to Claude Code. Adapt it for your chosen option.

---
*"I have a Dash application called portfolio_app.py. It already has two tabs. I want to add a risk summary below the CAPM regression table in Tab 1. Add a new section with three clearly labelled rows: 95% Historical VaR (1-day), 95% CVaR (1-day), and Maximum Drawdown. Compute these from the same portfolio daily return series already used in the file. Display them in a Dash DataTable. Format the values as percentages with 2 decimal places. Colour any negative value red. Add a brief plain-English explanation above the table saying what VaR and CVaR mean."*

---

## Presentation Scaffold (5 minutes)

1. **What the app does and who it is for** (1 min): describe portfolio_app.py as if presenting to a risk committee
2. **Live demo** (2 min): show the original tabs + your extension
3. **Investment insight** (1 min): what decision does your extension make easier?
4. **What you would add next** (1 min): one more feature, if you had another week

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CHALLENGE A — STARTER CELL
# This cell reproduces the key charts from portfolio_app.py as standalone
# Plotly figures. You do not need to run the app server to see these — they
# appear directly in this notebook.
# ══════════════════════════════════════════════════════════════════════════════

print('=== CHALLENGE A: PORTFOLIO VIEWER STARTER ===')
print()

# ── TAB 1 PREVIEW: Cumulative return chart ───────────────────────────────────
fig_a1 = go.Figure()   # create a new blank figure

# Add the portfolio cumulative return line
fig_a1.add_trace(
    go.Scatter(
        x    = cumulative_portfolio.index,
        y    = cumulative_portfolio,
        name = 'GIS Portfolio',
        mode = 'lines',
        line = dict(width=2.5, color='steelblue'),
    )
)

# Add the benchmark cumulative return line for comparison
fig_a1.add_trace(
    go.Scatter(
        x    = cumulative_benchmark.index,
        y    = cumulative_benchmark,
        name = 'S&P 500 Benchmark',
        mode = 'lines',
        line = dict(width=2, color='crimson', dash='dash'),   # dashed red = benchmark
    )
)

# Add reference line at 100 (the starting investment value)
fig_a1.add_hline(y=100, line_dash='dot', line_color='grey', line_width=1)

fig_a1.update_layout(
    title       = 'Tab 1 Preview — Cumulative Return (portfolio_app.py style)',
    xaxis_title = 'Date',
    yaxis_title = 'Value (Base = 100)',
    height      = 400,
    template    = 'plotly_white',
)

fig_a1.show()   # display the chart inline

# ── CAPM REGRESSION (Tab 1 preview) ──────────────────────────────────────────
# CAPM: Portfolio Return = Alpha + Beta × Benchmark Return
# Beta > 1 means the portfolio amplifies market moves
# Alpha > 0 means the portfolio earns more than the market explains
# We use monthly returns for stability (daily returns are noisier)

monthly_port  = monthly_returns.dot(weights)   # monthly portfolio return (equal-weighted)
monthly_bench = (
    benchmark
    .resample('ME')     # group to month-end
    .last()             # take the last trading day price
    .pct_change()       # month-over-month return
    .dropna()           # drop the first month
    .squeeze()          # convert from single-column DataFrame to Series
)

# Align the portfolio and benchmark monthly returns to the same date index
aligned = pd.concat([monthly_port, monthly_bench], axis=1, join='inner')
aligned.columns = ['Portfolio', 'Benchmark']   # rename columns for clarity

# Run the OLS regression: slope = beta, intercept = alpha (monthly)
slope, intercept, r_value, p_value, std_err = stats.linregress(
    aligned['Benchmark'],   # x: monthly benchmark return
    aligned['Portfolio'],   # y: monthly portfolio return
)

alpha = intercept * 12   # annualise alpha: monthly intercept × 12 months per year
beta  = slope            # beta is already dimensionless

# Build a summary table of CAPM results
capm_table = pd.DataFrame({
    'Metric': ['Alpha (annualised)', 'Beta', 'R²', 'p-value (beta)', 'Observations'],
    'Value':  [
        f'{alpha:.4%}',         # alpha formatted as a percentage
        f'{beta:.4f}',          # beta formatted to 4 decimal places
        f'{r_value**2:.4f}',    # R-squared: proportion of portfolio variance explained by market
        f'{p_value:.4f}',       # p-value for the null hypothesis that beta = 0
        f'{len(aligned)}',      # number of monthly observations
    ],
})

print('CAPM Regression Results:')
print(capm_table.to_string(index=False))   # print without the row number index
print()
print(f'Interpretation:')
print(f'  Beta = {beta:.2f}: for every 1% the market moves, the portfolio moves ~{beta:.2f}%')
print(f'  Alpha = {alpha:.2%} (annualised): return above/below what beta would predict')
print(f'  R² = {r_value**2:.2f}: {r_value**2*100:.0f}% of portfolio return is explained by the benchmark')
print()
print('=== NEXT STEP ===')
print('Choose one of the three extension options above.')
print('Brief Claude Code using the suggested brief (adapted for your chosen option).')
print('Iterate until the app looks the way you want it.')

---
---

# ══════════════════════════════════════
# CHALLENGE B — Interrogate the Momentum Strategy
# ══════════════════════════════════════

## What `momentum_investing.ipynb` Already Does

The momentum notebook implements a classic **12-1 cross-sectional momentum strategy** across S&P 500 constituents:

- **Universe:** All current S&P 500 constituents (downloaded from Wikipedia)
- **Signal:** 12-month return, excluding the most recent month (12-1 momentum)
- **Portfolio:** Top K stocks by momentum score, equal-weighted, monthly rebalancing
- **History:** 2002 to today (25 years of data)
- **Configurable K:** Run with top 5, 10, or 20 stocks and compare

**What it produces:**
- Annualised return, volatility, Sharpe ratio, maximum drawdown
- Cumulative return chart (momentum vs equal-weight vs SPY)
- Monthly returns heatmap
- Drawdown chart
- Turnover analysis
- Sector exposure over time
- K=5/10/20 performance comparison
- Rolling 36-month Sharpe ratio

## Your Challenge — Pick ONE Extension

Run the momentum notebook (or the starter cell below), then direct Claude Code to add one of these:

**Option 1: K-Comparison Chart**
Build a 2×2 subplot grid comparing K=5, K=10, K=20 across four metrics: annualised return, volatility, Sharpe, and max drawdown. Use a grouped bar chart for each metric. Colour each K value consistently across all four panels.

**Option 2: Sector Concentration Metric**
The strategy's sector exposure chart shows which sectors dominate the portfolio over time. Add a Herfindahl-Hirschman Index (HHI) of sector weights below it — a single number that measures how concentrated the sector exposure is. HHI = sum of squared sector weights. HHI of 1.0 = 100% in one sector; HHI of 1/N = perfectly diversified.

**Option 3: Momentum Decay Trend Line**
The rolling 36-month Sharpe chart shows whether momentum has become weaker over time. Add: (1) a horizontal reference line at the full-period average Sharpe; (2) a linear trend line through the rolling Sharpe series; (3) a text annotation stating whether momentum has strengthened or weakened over the observed period.

## Suggested Claude Code Brief (Option 1)

---
*"I have a Python dictionary called `results` where each key is a K value (5, 10, or 20) and each value is a dict with four keys: 'return' (annualised), 'volatility' (annualised), 'sharpe', 'max_drawdown'. Build a 2×2 subplot grid using Plotly where each panel is a grouped bar chart comparing the three K values on one metric. Panel 1: annualised return. Panel 2: volatility. Panel 3: Sharpe ratio. Panel 4: max drawdown. Use blue for K=5, orange for K=10, green for K=20. Add a clear title to each panel. Make the whole figure 1000×700 pixels. Format the y-axis as a percentage for return, volatility, and max drawdown; leave Sharpe as a plain number."*

---

## Presentation Scaffold (5 minutes)

1. **What momentum investing is and why it has worked** (1 min): explain the 12-1 strategy in plain English
2. **Live demo** (2 min): show the key cumulative return chart + your chosen extension
3. **Investment insight from your extension** (1 min): what does it tell you about how to use this strategy?
4. **Known weaknesses** (1 min): momentum crashes, survivorship bias, transaction costs — how would you manage these?

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CHALLENGE B — MOMENTUM STRATEGY STARTER CELL
# This cell runs the data pipeline from momentum_investing.ipynb:
#   1. Downloads S&P 500 constituents from Wikipedia
#   2. Downloads 3 years of price data (shorter than the full 25-year backtest)
#   3. Computes monthly returns
#   4. Runs a quick backtest for K=5, 10, 20 and stores results in `results`
#
# This gives you working data in ~60 seconds instead of the full 5-10 minutes
# that the 25-year backtest requires.
# Once you have `results`, you can brief Claude Code on your chosen extension.
# ══════════════════════════════════════════════════════════════════════════════

print('=== CHALLENGE B: MOMENTUM STRATEGY STARTER ===')
print()
print('Step 1: Downloading S&P 500 constituents from Wikipedia...')

# Download the current S&P 500 constituent list from Wikipedia
# pd.read_html() parses HTML tables from a URL and returns a list of DataFrames
sp500_table = pd.read_html(
    'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies',
    header=0,      # first row of the table is the column header
)[0]               # [0] = take the first table found on the page

# Extract just the ticker symbols as a Python list
sp500_tickers = (
    sp500_table['Symbol']                           # the 'Symbol' column contains ticker codes
    .str.replace('.', '-', regex=False)             # Wikipedia uses '.' (BRK.B); yfinance uses '-' (BRK-B)
    .tolist()                                       # convert the Series to a plain Python list
)

print(f'  Found {len(sp500_tickers)} S&P 500 constituents')
print()

# Use a 3-year window for speed — the full backtest uses 25 years
mom_end   = date.today()
mom_start = date(mom_end.year - 3, mom_end.month, mom_end.day)

print(f'Step 2: Downloading prices for {len(sp500_tickers)} tickers ({mom_start} → {mom_end})...')
print('  This may take 60–90 seconds...')

# Download all S&P 500 prices in one batch — yfinance is efficient for large ticker lists
sp500_raw = yf.download(
    tickers     = sp500_tickers,
    start       = mom_start,
    end         = mom_end,
    auto_adjust = True,
    progress    = False,     # suppress the progress bar for cleaner output
    threads     = True,      # use multi-threading for faster download
)

sp500_prices = (
    sp500_raw['Close']            # adjusted close prices
    .ffill()                      # forward-fill any missing values
    .dropna(axis=1, how='all')    # drop tickers where ALL values are missing (delistings, errors)
)

print(f'  Downloaded {sp500_prices.shape[1]} tickers × {sp500_prices.shape[0]} trading days')
print()

# Step 3: Compute monthly returns from daily prices
print('Step 3: Computing monthly returns...')

sp500_monthly_prices = (
    sp500_prices
    .resample('ME')   # group daily prices into calendar months
    .last()           # take the last available trading day price as the month-end price
)

sp500_monthly_returns = (
    sp500_monthly_prices
    .pct_change()       # month-over-month simple return
    .dropna(how='all')  # remove months where all tickers have NaN (typically the first month)
)

print(f'  Monthly returns shape: {sp500_monthly_returns.shape}')
print()

# Step 4: Run a quick 12-1 momentum backtest for K = 5, 10, 20
print('Step 4: Running 12-1 momentum backtest (K = 5, 10, 20)...')

def run_momentum_backtest(monthly_returns, K, formation=12, skip=1):
    '''
    12-1 momentum: rank stocks by their return over the past `formation` months,
    skipping the most recent `skip` months. Go long the top K stocks each month.
    Returns a Series of equal-weighted portfolio monthly returns.

    formation: how many months to measure momentum over (standard = 12)
    skip: how many recent months to exclude from the signal (standard = 1, avoids reversal)
    K: how many top stocks to hold in the portfolio
    '''
    portfolio_returns = []   # list to collect monthly portfolio returns
    dates             = []   # list to collect the corresponding dates

    # We need at least `formation + skip` months of history before we can compute the signal
    for i in range(formation + skip, len(monthly_returns)):
        # The signal window: from [i - formation - skip] to [i - skip - 1]
        signal_end   = i - skip              # end of the signal window (exclude the last `skip` months)
        signal_start = signal_end - formation   # start of the signal window

        signal_window = monthly_returns.iloc[signal_start:signal_end]   # slice the relevant months

        # Compute the cumulative return over the signal window for each stock
        cum_returns = (1 + signal_window).prod() - 1   # compound the monthly returns
        cum_returns = cum_returns.dropna()              # drop stocks with insufficient data

        if len(cum_returns) < K:
            continue   # skip this month if fewer than K stocks have enough data

        # Select the top K stocks by momentum score (highest past return)
        top_k = cum_returns.nlargest(K).index.tolist()

        # Compute the equal-weighted portfolio return in the HOLDING month (month i)
        holding_returns = monthly_returns.iloc[i][top_k].dropna()
        if len(holding_returns) == 0:
            continue   # skip if all selected stocks have missing data in the holding month

        portfolio_return = holding_returns.mean()   # equal-weight = simple average of returns
        portfolio_returns.append(portfolio_return)
        dates.append(monthly_returns.index[i])

    return pd.Series(portfolio_returns, index=dates)   # return a Series indexed by date


# Run the backtest for three values of K and store results in a dictionary
results = {}   # will hold {'return', 'volatility', 'sharpe', 'max_drawdown', 'returns'} for each K

for K in [5, 10, 20]:
    port_returns = run_momentum_backtest(sp500_monthly_returns, K=K)   # run the backtest for this K

    ann_ret  = port_returns.mean() * 12              # annualised return: average monthly × 12
    ann_vol  = port_returns.std() * np.sqrt(12)      # annualised volatility: monthly std × sqrt(12)
    sharpe   = ann_ret / ann_vol                     # simplified Sharpe (no risk-free rate for speed)
    cum      = (1 + port_returns).cumprod()          # cumulative return index
    mdd      = (cum / cum.cummax() - 1).min()        # maximum drawdown

    results[K] = {
        'returns'      : port_returns,    # the monthly return Series (for charts)
        'return'       : ann_ret,         # scalar: annualised return
        'volatility'   : ann_vol,         # scalar: annualised volatility
        'sharpe'       : sharpe,          # scalar: annualised Sharpe ratio
        'max_drawdown' : mdd,             # scalar: maximum drawdown (negative)
    }

    print(f'  K={K:<3}  Return: {ann_ret:.2%}  Vol: {ann_vol:.2%}  Sharpe: {sharpe:.2f}  MaxDD: {mdd:.2%}')

print()
print('`results` dictionary is ready. Use it in your Claude Code brief.')
print()

# Quick comparison chart — show cumulative return for K=5, 10, 20
fig_b = go.Figure()

for K in [5, 10, 20]:
    cum_returns = (1 + results[K]['returns']).cumprod() * 100   # index starting at 100
    fig_b.add_trace(
        go.Scatter(
            x    = cum_returns.index,
            y    = cum_returns.values,
            name = f'K={K}',
            mode = 'lines',
            line = dict(width=2),
        )
    )

fig_b.update_layout(
    title       = 'Momentum Strategy — Cumulative Return by K (3-Year, Quick Backtest)',
    xaxis_title = 'Date',
    yaxis_title = 'Value (Base = 100)',
    height      = 400,
    template    = 'plotly_white',
)
fig_b.show()

print()
print('=== NEXT STEP ===')
print('Choose one of the three extension options.')
print('Brief Claude Code using the suggested brief as a starting point.')

---
---

# ══════════════════════════════════════
# CHALLENGE C — The Macro Regime Dashboard
# ══════════════════════════════════════

## The Brief

Build a dashboard that classifies the current macroeconomic environment into one of four regimes:

| Regime | Signals | Asset Class Implication |
|--------|---------|------------------------|
| **Expansion** | Low CPI, positive curve, falling unemployment | Favour equities; cyclicals and financials |
| **Stagflation** | High CPI, inverted curve, rising unemployment | Favour commodities and inflation-linked bonds; reduce equities |
| **Recession** | Inverted curve, rising unemployment, falling CPI | Favour duration (long bonds), gold, defensives |
| **Recovery** | CPI contained, curve steepening, unemployment falling | Begin adding risk; small-caps and cyclicals outperform |

The dashboard should also show the 3 most similar historical periods (by distance from current macro conditions) and how major asset classes performed in the 12 months following those periods.

## Suggested Claude Code Brief

---
*"I have a pandas DataFrame called `macro_df` with monthly columns: CPI_YoY (%), FFR (%), Spread_2Y10Y (pp), and Unemployment (%). I also have a Series called `current_macro` with the current values of each indicator.*

*Build a Dash app with two panels:*
*Panel 1: Four metric cards showing the current value of each indicator. Each card should show the indicator name, the current value, and an arrow (▲ or ▼) compared to the value 3 months ago. If CPI is above 4% or the spread is below 0, highlight that card in red. If unemployment is falling, highlight it in green.*

*Panel 2: A table showing the 3 most similar historical months (by Euclidean distance on normalised indicators), with columns: Date, CPI YoY, Fed Funds Rate, 2Y/10Y Spread, Unemployment, and the total return of SPY, IEF, and GLD in the 12 months following that period. Use yfinance to compute those asset returns.*

*Add a regime classification label at the top based on simple rules: Expansion if spread > 0 and CPI < 3; Stagflation if CPI > 4 and spread < 0; Recession if spread < 0 and unemployment rising; Recovery otherwise."*

---

## Presentation Scaffold (5 minutes)

1. **Why macro context matters** (1 min): how do macro regimes affect asset class performance?
2. **Live demo** (2 min): show the dashboard — current conditions, regime label, historical analogues
3. **The investment insight** (1 min): what does the current regime suggest for GIS portfolio positioning?
4. **What you would improve** (1 min): what additional indicator would you add? What would make the "similar historical periods" more useful?

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CHALLENGE C — MACRO REGIME DASHBOARD STARTER CELL
# This cell builds the macro_df and current_macro objects that your Claude Code
# brief will reference. It also implements the regime classification and finds
# the 3 most similar historical periods.
# ══════════════════════════════════════════════════════════════════════════════

print('=== CHALLENGE C: MACRO REGIME DASHBOARD STARTER ===')
print()

if not macro_ok:
    print('ERROR: FRED data is unavailable. Check your API key in the setup cell and re-run.')
else:
    # ── BUILD macro_df ────────────────────────────────────────────────────────
    # Assemble a monthly DataFrame with 4 macro indicators
    # All series must be aligned to the same monthly date index
    macro_df = pd.DataFrame({
        'CPI_YoY'      : cpi_headline_yoy,                     # headline CPI year-over-year (%)
        'FFR'          : fed_funds.resample('ME').last(),       # Fed Funds Rate at month-end (%)
        'Spread_2Y10Y' : spread_series.resample('ME').last(),   # 2Y/10Y spread at month-end (pp)
        'Unemployment' : unemployment,                          # unemployment rate (%)
    })

    macro_df = (
        macro_df
        .dropna()      # remove months where any indicator is missing
        .sort_index()  # ensure rows are in chronological order
    )

    print(f'macro_df shape: {macro_df.shape}  (months × indicators)')
    print(f'Date range: {macro_df.index[0].date()}  →  {macro_df.index[-1].date()}')
    print()
    display(macro_df.tail(6))   # show the 6 most recent months

    # ── CURRENT MACRO VALUES ──────────────────────────────────────────────────
    current_macro = macro_df.iloc[-1]   # the most recent month's values

    print()
    print('=== CURRENT MACRO CONDITIONS ===')
    for indicator, value in current_macro.items():
        print(f'  {indicator:<20} {value:.2f}')

    # ── REGIME CLASSIFICATION ─────────────────────────────────────────────────
    # Simple rule-based classification using current indicator values
    cpi_now      = current_macro['CPI_YoY']          # current CPI year-over-year
    spread_now   = current_macro['Spread_2Y10Y']      # current 2Y/10Y spread
    unemp_now    = current_macro['Unemployment']      # current unemployment rate
    unemp_3m     = macro_df['Unemployment'].iloc[-4]  # unemployment rate 3 months ago
    unemp_rising = unemp_now > unemp_3m               # True if unemployment has risen in 3 months

    # Apply the regime rules in order of severity
    if cpi_now > 4.0 and spread_now < 0:
        regime = 'STAGFLATION'                      # high inflation AND inverted curve
    elif spread_now < 0 and unemp_rising:
        regime = 'LATE CYCLE / RECESSION RISK'      # inverted curve AND rising unemployment
    elif cpi_now < 3.0 and spread_now > 0 and not unemp_rising:
        regime = 'EXPANSION'                        # low inflation, positive curve, stable/falling unemployment
    elif not unemp_rising and cpi_now < 3.0:
        regime = 'RECOVERY'                         # conditions improving but not yet full expansion
    else:
        regime = 'MIXED SIGNALS'                    # conditions do not fit a clear regime

    print()
    print(f'=== REGIME: {regime} ===')

    # ── FIND 3 MOST SIMILAR HISTORICAL PERIODS ────────────────────────────────
    # We use Euclidean distance on normalised indicators to find the 3 historical
    # months that look most like today.
    # Normalisation (mean=0, std=1) ensures that indicators on different scales
    # (e.g. CPI in 0-10%, unemployment in 3-15%) are treated equally.

    macro_normalised = (
        macro_df
        .sub(macro_df.mean())   # subtract the historical mean of each column
        .div(macro_df.std())    # divide by the historical standard deviation of each column
    )

    current_normalised = (
        current_macro
        .sub(macro_df.mean())   # normalise using the HISTORICAL mean (not today's value)
        .div(macro_df.std())    # normalise using the HISTORICAL std
    )

    # Euclidean distance = sqrt(sum of squared differences across all 4 indicators)
    # Small distance = this historical month's macro conditions looked like today's
    distances = (
        macro_normalised
        .sub(current_normalised)   # difference between each historical month and today
        .pow(2)                    # square each difference
        .sum(axis=1)               # sum across the 4 indicators
        .pow(0.5)                  # take the square root → Euclidean distance
    )

    # Exclude the most recent 12 months to avoid trivially selecting "last month" as a match
    cutoff_date          = macro_df.index[-13]   # 12 months before the most recent month
    distances_historical = distances[distances.index < cutoff_date]   # only look at older data

    # Find the 3 smallest distances = the 3 most similar historical periods
    top3_dates = distances_historical.nsmallest(3).index

    print()
    print('=== 3 MOST SIMILAR HISTORICAL PERIODS ===')
    print('(by Euclidean distance on normalised macro indicators)')
    print()

    for rank, hist_date in enumerate(top3_dates, start=1):
        hist_vals   = macro_df.loc[hist_date]        # macro values in that historical month
        dist        = distances.loc[hist_date]       # Euclidean distance from today
        print(f'  #{rank}: {hist_date.date()}  (distance: {dist:.3f})')
        for indicator, value in hist_vals.items():
            current_val = current_macro[indicator]   # today's value for comparison
            diff        = value - current_val        # difference between historical and today
            print(f'         {indicator:<20} {value:.2f}  (today: {current_val:.2f},  diff: {diff:+.2f})')
        print()

    # ── QUICK VISUALISATION: MACRO DASHBOARD ─────────────────────────────────
    # Show all 4 macro indicators as a 2×2 chart grid for context
    fig_c = make_subplots(
        rows   = 2,
        cols   = 2,
        subplot_titles = ['CPI YoY (%)', 'Fed Funds Rate (%)', '2Y/10Y Spread (pp)', 'Unemployment (%)'],
    )

    # Panel 1: CPI year-over-year
    fig_c.add_trace(
        go.Scatter(
            x    = macro_df.index,
            y    = macro_df['CPI_YoY'],
            mode = 'lines',
            line = dict(width=2, color='steelblue'),
            name = 'CPI YoY',
            showlegend = False,
        ),
        row=1, col=1,
    )

    # Panel 2: Fed Funds Rate
    fig_c.add_trace(
        go.Scatter(
            x    = macro_df.index,
            y    = macro_df['FFR'],
            mode = 'lines',
            line = dict(width=2, color='darkorange'),
            name = 'FFR',
            showlegend = False,
        ),
        row=1, col=2,
    )

    # Panel 3: 2Y/10Y yield spread
    fig_c.add_trace(
        go.Scatter(
            x    = macro_df.index,
            y    = macro_df['Spread_2Y10Y'],
            mode = 'lines',
            line = dict(width=2, color='seagreen'),
            name = '2Y/10Y Spread',
            showlegend = False,
        ),
        row=2, col=1,
    )

    # Add zero reference line to the spread panel
    fig_c.add_hline(y=0, line_color='black', line_width=1, row=2, col=1)

    # Panel 4: Unemployment rate
    fig_c.add_trace(
        go.Scatter(
            x    = macro_df.index,
            y    = macro_df['Unemployment'],
            mode = 'lines',
            line = dict(width=2, color='crimson'),
            name = 'Unemployment',
            showlegend = False,
        ),
        row=2, col=2,
    )

    fig_c.update_layout(
        title    = dict(text=f'Macro Regime Indicators — Current Regime: {regime}', font=dict(size=16)),
        height   = 600,
        template = 'plotly_white',
    )

    fig_c.show()

    print()
    print('=== NEXT STEP ===')
    print('The `macro_df` and `current_macro` objects are ready.')
    print('Use the suggested Claude Code brief above to build the Dash dashboard.')
    print('Reference `macro_df`, `current_macro`, and `regime` directly in your brief.')

---
---

# ══════════════════════════════════════
# PRESENTATIONS & DEBRIEF
# ══════════════════════════════════════

## Presentation Order & Timing

Each group has **5 minutes** to present + **2 minutes** Q&A from the facilitator.

Suggested structure:
1. **What** (30 sec): What does your tool do? Who is it for?
2. **Demo** (2 min): Show it live — run the key cells or open the app
3. **Insight** (1 min): What investment decision does this make easier or better?
4. **Critique** (1 min): What did you have to simplify? What would you add?
5. **Brief** (30 sec): Read out the most effective Claude Code instruction you wrote

---

## Full-Group Debrief Discussion

The facilitator will lead a 15-minute discussion using these questions:

### On the tools:
- **Challenge A groups:** `portfolio_app.py` already exists and works. What would you add next? Who at GIS should be involved in specifying it? Is it ready to show a client?
- **Challenge B groups:** The momentum strategy has significant known weaknesses: momentum crashes (2009, 2020), survivorship bias, transaction costs at K=5. How would this tool need to change before you could present it to a client?
- **Challenge C groups:** The regime classification uses only 4 indicators. What would you add? How would you validate that the "similar historical periods" are genuinely useful?

### On Claude Code:
- What kinds of instructions worked well? What made them effective?
- What happened when the instruction was vague? What did Claude Code assume?
- How does briefing Claude Code compare to briefing a junior analyst?
- What could you do in the next 30 days using what you built today?

### On the seminar overall:
- Which session changed how you think about your current workflow?
- What is one thing you will do differently in the next two weeks?
- What would it take to implement one of these tools at GIS on a live basis?

---

## Thank You

This seminar has covered the full investment data workflow:

| Phase | Sessions | Key Skills |
|-------|----------|-----------|
| **ACQUIRE** | 1–2 | Download, inspect, clean, and compute returns from financial and macro data |
| **ANALYSE** | 3–4 | Portfolio construction, risk metrics, correlation, yield curve, inflation, FX |
| **ACT** | 5–6 | Interactive visualisation, dashboards, Claude Code-assisted development |

**All notebooks, data, and code from this seminar are yours to keep and adapt.**

The Python skills you have practised over these three days are not the most important takeaway. The most important takeaway is the *framework*: know what question you are asking, identify the right data source, compute the right metric, and communicate it in a format that drives a decision.

That framework — data → analysis → action — is the same whether you are working in Python, Excel, Bloomberg, or any other tool.

*Good luck, and please send feedback to the facilitator.*